In [30]:
import os, json, time, sqlite3, pickle, base64, requests
from datetime import datetime, timedelta
from email.mime.text import MIMEText
import pandas as pd

# ─── Google OAuth ───
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# ─── Paths ───
CREDS_FILE = "credentials.json"
TOKEN_FILE = "token.json"
DB_PATH = "knowledge_base.db"   # adjust if needed

# ─── Config ───
SCOPES = [
    "https://www.googleapis.com/auth/calendar",
    "https://www.googleapis.com/auth/gmail.send",
    "https://www.googleapis.com/auth/spreadsheets",
]

# ─── Your details (FILL THESE) ───
TEST_CALENDAR_ID = "cfiza8180@gmail.com"                  # ya test calendar ID
TEST_EMAIL_TO   = "cfiza8180@gmail.com"       # jahan email bhejni hai
CRM_SHEET_ID    = "1VtCun1Cf_jQFvGMv94q4g_cNzxZ6uXIdqTGh7iiEeWw"        # Google Sheet ID (URL se)
CRM_SHEET_NAME  = "Sheet1"
N8N_WEBHOOK_URL = "http://localhost:5678/webhook-test/6901482b-d1ac-4d50-8a07-024ca415a718"
# ─── Business hours ───
OFFICE_START = 10   # 10 AM
OFFICE_END   = 19   # 7 PM

print("Config loaded ✅")
print("DB exists:", os.path.exists(DB_PATH))
print("Credentials exists:", os.path.exists(CREDS_FILE))

Config loaded ✅
DB exists: True
Credentials exists: True


In [2]:
conn = sqlite3.connect(DB_PATH)
print("Properties:", pd.read_sql_query("SELECT COUNT(*) n FROM properties", conn)["n"][0])
print("Agents:", pd.read_sql_query("SELECT COUNT(*) n FROM agents", conn)["n"][0])

Properties: 500
Agents: 7


## 📅 Section 2: Task 1 — Google Calendar

In [3]:
def get_google_creds():
    """Load or refresh Google credentials."""
    creds = None
    if os.path.exists(TOKEN_FILE):
        creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDS_FILE, SCOPES)
            creds = flow.run_local_server(port=8080)   # <-- FIXED
        with open(TOKEN_FILE, "w") as f:
            f.write(creds.to_json())
    return creds

creds = get_google_creds()
calendar_service = build("calendar", "v3", credentials=creds)
print("Calendar service ready ✅")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=128860445196-r7me4q5lq1ba6ujfjmkj1a1f7qj0o2qo.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fspreadsheets&state=sQEUMc8jM17b2cWqncdSDt8oj81cfO&code_challenge=XeyjD9XXz4cbk7NsQ5MYgEzMwOQM1Z7aHDkwxD3OiSU&code_challenge_method=S256&access_type=offline
Calendar service ready ✅


In [4]:
def create_calendar_event(client_name, client_phone, employee_email,
                          property_title, start_dt, end_dt, notes=""):
    """
    Create a calendar event.
    start_dt / end_dt: datetime objects
    Returns: event ID if successful, None if failed
    """
    event = {
        "summary": f"Property Visit — {client_name}",
        "location": property_title,
        "description": (
            f"Client: {client_name}\n"
            f"Phone: {client_phone}\n"
            f"Property: {property_title}\n"
            f"Notes: {notes}"
        ),
        "start": {"dateTime": start_dt.isoformat(), "timeZone": "Asia/Karachi"},
        "end":   {"dateTime": end_dt.isoformat(),   "timeZone": "Asia/Karachi"},
        "attendees": [{"email": employee_email}],
        "reminders": {
            "useDefault": False,
            "overrides": [
                {"method": "email", "minutes": 60},
                {"method": "popup", "minutes": 15},
            ],
        },
    }

    try:
        created = calendar_service.events().insert(
            calendarId=TEST_CALENDAR_ID, body=event
        ).execute()
        return created["id"], created.get("htmlLink")
    except Exception as e:
        print(f"❌ Calendar error: {e}")
        return None, None

print("create_calendar_event ready ✅")

create_calendar_event ready ✅


In [5]:
# Book kal ke liye ek test event
tomorrow = datetime.now() + timedelta(days=1)
start_dt = tomorrow.replace(hour=15, minute=0, second=0, microsecond=0)
end_dt   = start_dt + timedelta(hours=1)

event_id, event_link = create_calendar_event(
    client_name="Test Client",
    client_phone="0300-1234567",
    employee_email=TEST_EMAIL_TO,
    property_title="5 Marla House, DHA Phase 6 Lahore",
    start_dt=start_dt,
    end_dt=end_dt,
    notes="Demo booking test",
)

if event_id:
    print(f"✅ Event created!")
    print(f"   ID: {event_id}")
    print(f"   Link: {event_link}")
else:
    print("❌ Event creation failed")

✅ Event created!
   ID: 0r5qatu38mpp62m1e3c4isbkas
   Link: https://www.google.com/calendar/event?eid=MHI1cWF0dTM4bXBwNjJtMWUzYzRpc2JrYXMgY2ZpemE4MTgwQG0


In [6]:
def check_availability(date, hour_start=OFFICE_START, hour_end=OFFICE_END):
    """
    Return list of free 1-hour slots on given date.
    date: datetime.date or datetime object (date part used)
    """
    if isinstance(date, datetime):
        date = date.date()

    day_start = datetime.combine(date, datetime.min.time()).replace(hour=hour_start)
    day_end   = datetime.combine(date, datetime.min.time()).replace(hour=hour_end)

    try:
        events_result = calendar_service.events().list(
            calendarId=TEST_CALENDAR_ID,
            timeMin=day_start.isoformat() + "Z",
            timeMax=day_end.isoformat() + "Z",
            singleEvents=True,
            orderBy="startTime",
        ).execute()
        events = events_result.get("items", [])
    except Exception as e:
        print(f"❌ Availability error: {e}")
        return []

    # Block booked hours
    booked_hours = set()
    for ev in events:
        s = ev["start"].get("dateTime")
        if s:
            booked_hours.add(datetime.fromisoformat(s.replace("Z", "")).hour)

    free_slots = []
    for h in range(hour_start, hour_end):
        if h not in booked_hours:
            free_slots.append(f"{h:02d}:00")

    return free_slots

# Test — kal ke free slots
test_slots = check_availability(tomorrow)
print(f"Free slots on {tomorrow.date()}: {test_slots}")

Free slots on 2026-09-26: ['11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00']


## 📧 Section 3: Task 2 — Email Automation

In [7]:
gmail_service = build("gmail", "v1", credentials=creds)

def send_email(to_email, subject, body):
    """Send email via Gmail API."""
    message = MIMEText(body)
    message["to"] = to_email
    message["subject"] = subject

    raw = base64.urlsafe_b64encode(message.as_bytes()).decode()

    try:
        sent = gmail_service.users().messages().send(
            userId="me", body={"raw": raw}
        ).execute()
        return sent["id"]
    except Exception as e:
        print(f"❌ Email error: {e}")
        return None

print("send_email ready ✅")

send_email ready ✅


In [8]:
email_body = """Assalam-o-Alaikum,

Aap ki property visit schedule ho gayi hai.

Client: Test Client
Property: 5 Marla House, DHA Phase 6 Lahore
Date: {}
Time: 3:00 PM - 4:00 PM
Phone: 0300-1234567

Please confirm attendance.

Regards,
RealEstate Hub
""".format(tomorrow.date())

msg_id = send_email(
    to_email=TEST_EMAIL_TO,
    subject="[RealEstate Hub] New Property Visit Booked",
    body=email_body,
)

print(f"✅ Email sent!" if msg_id else "❌ Failed")
print(f"   Message ID: {msg_id}")

✅ Email sent!
   Message ID: 1a0d546923fc6324


## 📋 Section 4: Task 3 — Appointment Management

In [9]:
cur = conn.cursor()
cur.executescript("""
CREATE TABLE IF NOT EXISTS appointments (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    client_name TEXT,
    client_phone TEXT,
    employee_email TEXT,
    property_title TEXT,
    start_time TEXT,
    end_time TEXT,
    calendar_event_id TEXT,
    status TEXT DEFAULT 'booked',
    created_at TEXT
);

CREATE TABLE IF NOT EXISTS crm_calls (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    client_phone TEXT,
    client_name TEXT,
    intent TEXT,
    transcript_summary TEXT,
    appointment_id INTEGER,
    created_at TEXT
);
""")
conn.commit()
print("CRM tables ready ✅")

CRM tables ready ✅


In [10]:
def book_appointment(client_name, client_phone, employee_email,
                     property_title, start_dt):
    """
    Book appointment end-to-end:
      1. Check calendar availability
      2. Create calendar event
      3. Send email to employee
      4. Log to DB
    Returns dict with status.
    """
    result = {"success": False, "stage": None, "details": {}}

    # 1. Availability check
    free_slots = check_availability(start_dt)
    slot_str = f"{start_dt.hour:02d}:00"
    if slot_str not in free_slots:
        result["stage"] = "availability"
        result["details"] = {"reason": f"Slot {slot_str} not free", "free": free_slots}
        return result

    # 2. Create calendar event
    end_dt = start_dt + timedelta(hours=1)
    event_id, event_link = create_calendar_event(
        client_name, client_phone, employee_email,
        property_title, start_dt, end_dt,
    )
    if not event_id:
        result["stage"] = "calendar"
        result["details"] = {"reason": "Calendar event creation failed"}
        return result

    # 3. Email employee
    email_body = f"""Assalam-o-Alaikum,

New property visit scheduled.

Client : {client_name}
Phone  : {client_phone}
Property: {property_title}
Date   : {start_dt.date()}
Time   : {start_dt.strftime('%I:%M %p')} - {end_dt.strftime('%I:%M %p')}

Calendar link: {event_link}

Regards,
RealEstate Hub
"""
    msg_id = send_email(
        employee_email,
        f"[RealEstate Hub] Visit — {client_name} on {start_dt.date()}",
        email_body,
    )

    # 4. Save to DB
    cur.execute("""
        INSERT INTO appointments
        (client_name, client_phone, employee_email, property_title,
         start_time, end_time, calendar_event_id, status, created_at)
        VALUES (?,?,?,?,?,?,?,?,?)
    """, (
        client_name, client_phone, employee_email, property_title,
        start_dt.isoformat(), end_dt.isoformat(), event_id,
        "booked", datetime.now().isoformat(),
    ))
    conn.commit()
    appt_id = cur.lastrowid

    result["success"] = True
    result["stage"] = "complete"
    result["details"] = {
        "appointment_id": appt_id,
        "calendar_event_id": event_id,
        "calendar_link": event_link,
        "email_message_id": msg_id,
        "start": start_dt.isoformat(),
    }
    return result

print("book_appointment ready ✅")

book_appointment ready ✅


In [11]:
def reschedule_appointment(appointment_id, new_start_dt):
    """Reschedule existing appointment."""
    row = pd.read_sql_query(
        "SELECT * FROM appointments WHERE id=?",
        conn, params=[appointment_id]
    )
    if row.empty:
        return {"success": False, "reason": "appointment not found"}

    row = row.iloc[0]

    # 1. Check new slot availability
    free_slots = check_availability(new_start_dt)
    if f"{new_start_dt.hour:02d}:00" not in free_slots:
        return {"success": False, "reason": "new slot not free", "free": free_slots}

    # 2. Update calendar event
    new_end_dt = new_start_dt + timedelta(hours=1)
    try:
        calendar_service.events().patch(
            calendarId=TEST_CALENDAR_ID,
            eventId=row["calendar_event_id"],
            body={
                "start": {"dateTime": new_start_dt.isoformat(), "timeZone": "Asia/Karachi"},
                "end":   {"dateTime": new_end_dt.isoformat(),   "timeZone": "Asia/Karachi"},
            },
        ).execute()
    except Exception as e:
        return {"success": False, "reason": f"calendar update failed: {e}"}

    # 3. Notify employee
    send_email(
        row["employee_email"],
        f"[RealEstate Hub] RESCHEDULED — {row['client_name']}",
        f"Visit rescheduled.\n\nNew time: {new_start_dt}\nProperty: {row['property_title']}",
    )

    # 4. Update DB
    cur.execute("""
        UPDATE appointments SET start_time=?, end_time=?, status='rescheduled'
        WHERE id=?
    """, (new_start_dt.isoformat(), new_end_dt.isoformat(), appointment_id))
    conn.commit()

    return {"success": True, "new_start": new_start_dt.isoformat()}

print("reschedule_appointment ready ✅")

reschedule_appointment ready ✅


In [12]:
def cancel_appointment(appointment_id, reason="Client requested"):
    """Cancel an appointment."""
    row = pd.read_sql_query(
        "SELECT * FROM appointments WHERE id=?",
        conn, params=[appointment_id]
    )
    if row.empty:
        return {"success": False, "reason": "appointment not found"}

    row = row.iloc[0]

    # 1. Delete calendar event
    try:
        calendar_service.events().delete(
            calendarId=TEST_CALENDAR_ID,
            eventId=row["calendar_event_id"],
        ).execute()
    except Exception as e:
        print(f"⚠️ Calendar delete failed: {e}")

    # 2. Notify employee
    send_email(
        row["employee_email"],
        f"[RealEstate Hub] CANCELLED — {row['client_name']}",
        f"Visit cancelled.\n\nReason: {reason}\nOriginal time: {row['start_time']}",
    )

    # 3. Update DB
    cur.execute(
        "UPDATE appointments SET status='cancelled' WHERE id=?",
        (appointment_id,)
    )
    conn.commit()

    return {"success": True, "cancelled_id": appointment_id}

print("cancel_appointment ready ✅")

cancel_appointment ready ✅


In [13]:
# 1. BOOK — parson ke liye
day_after = datetime.now() + timedelta(days=2)
book_start = day_after.replace(hour=11, minute=0, second=0, microsecond=0)

book_result = book_appointment(
    client_name="Ali Raza",
    client_phone="0300-9876543",
    employee_email=TEST_EMAIL_TO,
    property_title="10 Marla House, Bahria Town Lahore",
    start_dt=book_start,
)
print("📅 BOOK:", json.dumps(book_result, indent=2, default=str))

# 2. RESCHEDULE — 4 baje shift karo
if book_result["success"]:
    appt_id = book_result["details"]["appointment_id"]
    new_time = book_start.replace(hour=16)
    resched_result = reschedule_appointment(appt_id, new_time)
    print("\n🔄 RESCHEDULE:", json.dumps(resched_result, indent=2, default=str))

    # 3. CANCEL — usi appointment ko cancel karo
    cancel_result = cancel_appointment(appt_id, reason="Test cleanup")
    print("\n❌ CANCEL:", json.dumps(cancel_result, indent=2, default=str))

📅 BOOK: {
  "success": true,
  "stage": "complete",
  "details": {
    "appointment_id": 1,
    "calendar_event_id": "1tc52mct3k69uru1vd5e7j1e84",
    "calendar_link": "https://www.google.com/calendar/event?eid=MXRjNTJtY3QzazY5dXJ1MXZkNWU3ajFlODQgY2ZpemE4MTgwQG0",
    "email_message_id": "1a0d54886f3edda2",
    "start": "2026-09-27T11:00:00"
  }
}

🔄 RESCHEDULE: {
  "success": true,
  "new_start": "2026-09-27T16:00:00"
}

❌ CANCEL: {
  "success": true,
  "cancelled_id": 1
}


## 🔄 Section 5: Task 4 — n8n Workflow

In [14]:
def trigger_n8n_workflow(payload):
    """
    Send event to n8n webhook.
    n8n workflow: Call → Intent → Match → Appointment → Calendar → Email → CRM
    """
    try:
        r = requests.post(
            N8N_WEBHOOK_URL,
            json=payload,
            timeout=10,
        )
        return {"status": r.status_code, "response": r.text[:200]}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

print("trigger_n8n_workflow ready ✅")
print(f"Webhook URL: {N8N_WEBHOOK_URL}")

trigger_n8n_workflow ready ✅
Webhook URL: http://localhost:5678/webhook/6901482b-d1ac-4d50-8a07-024ca415a718


In [15]:
def full_workflow(client_name, client_phone, employee_email,
                  property_title, start_dt):
    """
    End-to-end workflow:
      Intent → Match → Appointment → Calendar → Email → CRM → n8n notify
    """
    log = {"steps": []}

    # Step 1: Intent (assume 'book_appointment')
    log["steps"].append({"step": "intent", "value": "book_appointment"})

    # Step 2: Book (covers calendar + email)
    book_result = book_appointment(
        client_name, client_phone, employee_email, property_title, start_dt
    )
    log["steps"].append({"step": "book", "result": book_result})

    if not book_result["success"]:
        return log

    # Step 3: CRM log
    crm_result = log_to_crm(
        client_phone=client_phone,
        client_name=client_name,
        intent="book_appointment",
        summary=f"Booked {property_title} at {start_dt}",
        appointment_id=book_result["details"]["appointment_id"],
    )
    log["steps"].append({"step": "crm", "result": crm_result})

    # Step 4: n8n notify
    n8n_result = trigger_n8n_workflow({
        "event": "appointment_booked",
        "client": client_name,
        "phone": client_phone,
        "property": property_title,
        "start": start_dt.isoformat(),
        "appointment_id": book_result["details"]["appointment_id"],
    })
    log["steps"].append({"step": "n8n", "result": n8n_result})

    return log

## 📊 Section 6: Task 5 — CRM Logging (Google Sheets)

In [16]:
sheets_service = build("sheets", "v4", credentials=creds)

def _ensure_sheet_headers():
    """Make sure CRM sheet has headers."""
    try:
        result = sheets_service.spreadsheets().values().get(
            spreadsheetId=CRM_SHEET_ID,
            range=f"{CRM_SHEET_NAME}!A1:H1",
        ).execute()
        if "values" not in result:
            # Add headers
            headers = [[
                "timestamp", "client_phone", "client_name",
                "intent", "summary", "appointment_id",
                "agent_name", "notes"
            ]]
            sheets_service.spreadsheets().values().update(
                spreadsheetId=CRM_SHEET_ID,
                range=f"{CRM_SHEET_NAME}!A1",
                valueInputOption="RAW",
                body={"values": headers},
            ).execute()
    except Exception as e:
        print(f"⚠️ Sheet header check failed: {e}")

_ensure_sheet_headers()
print("CRM sheet ready ✅")

CRM sheet ready ✅


In [17]:
def log_to_crm(client_phone, client_name, intent, summary,
               appointment_id=None, agent_name="", notes=""):
    """
    Log call details to Google Sheets (CRM) and SQLite (local backup).
    """
    timestamp = datetime.now().isoformat()
    row = [[
        timestamp, client_phone, client_name, intent,
        summary, str(appointment_id) if appointment_id else "",
        agent_name, notes,
    ]]

    # 1. Google Sheets
    sheet_status = "ok"
    try:
        sheets_service.spreadsheets().values().append(
            spreadsheetId=CRM_SHEET_ID,
            range=f"{CRM_SHEET_NAME}!A:H",
            valueInputOption="RAW",
            body={"values": row},
        ).execute()
    except Exception as e:
        sheet_status = f"error: {e}"

    # 2. SQLite backup
    cur.execute("""
        INSERT INTO crm_calls
        (client_phone, client_name, intent, transcript_summary,
         appointment_id, created_at)
        VALUES (?,?,?,?,?,?)
    """, (client_phone, client_name, intent, summary,
          appointment_id, timestamp))
    conn.commit()

    return {"sheets_status": sheet_status, "sqlite_id": cur.lastrowid}

print("log_to_crm ready ✅")

log_to_crm ready ✅


In [18]:
crm_result = log_to_crm(
    client_phone="0300-1111222",
    client_name="Sara Ahmed",
    intent="buyer_inquiry",
    summary="Interested in 5 Marla DHA Phase 6. Budget 2.5 crore.",
    appointment_id=None,
    agent_name="Ahmed",
    notes="Callback requested for Friday",
)
print("📝 CRM:", json.dumps(crm_result, indent=2))

📝 CRM: {
  "sheets_status": "ok",
  "sqlite_id": 1
}


In [19]:
def get_customer_history(phone):
    """Fetch all past interactions for a phone number."""
    df = pd.read_sql_query(
        "SELECT * FROM crm_calls WHERE client_phone=? ORDER BY created_at DESC",
        conn, params=[phone]
    )
    return df

# Test
hist = get_customer_history("0300-1111222")
display(hist)

,id,client_phone,client_name,intent,transcript_summary,appointment_id,created_at
0,1,0300-1111222,Sara Ahmed,buyer_inquiry,Interested in 5 Marla DHA Phase 6. Budget 2.5 ...,None,2026-09-25T02:20:35.786873


In [31]:
# End-to-end test
day_after = datetime.now() + timedelta(days=2)
test_start = day_after.replace(hour=14, minute=0, second=0, microsecond=0)

workflow_log = full_workflow(
    client_name="Bilal Khan",
    client_phone="0300-5556667",
    employee_email=TEST_EMAIL_TO,
    property_title="1 Kanal House, DHA Phase 5 Lahore",
    start_dt=test_start,
)

print(json.dumps(workflow_log, indent=2, default=str))

{
  "steps": [
    {
      "step": "intent",
      "value": "book_appointment"
    },
    {
      "step": "book",
      "result": {
        "success": true,
        "stage": "complete",
        "details": {
          "appointment_id": 5,
          "calendar_event_id": "lcjvkvkghd7afnu9tm55l9u8lo",
          "calendar_link": "https://www.google.com/calendar/event?eid=bGNqdmt2a2doZDdhZm51OXRtNTVsOXU4bG8gY2ZpemE4MTgwQG0",
          "email_message_id": "1a0d55058f6f18e0",
          "start": "2026-09-27T14:00:00"
        }
      }
    },
    {
      "step": "crm",
      "result": {
        "sheets_status": "ok",
        "sqlite_id": 5
      }
    },
    {
      "step": "n8n",
      "result": {
        "status": 200,
        "response": "{\"message\":\"Workflow was started\"}"
      }
    }
  ]
}


## 📁 Section 7: Summary + Save

In [22]:
appts = pd.read_sql_query("SELECT * FROM appointments", conn)
appts.to_csv("appointments.csv", index=False)
print(f"Saved {len(appts)} appointments to appointments.csv")
display(appts.tail())

Saved 2 appointments to appointments.csv


,id,client_name,client_phone,employee_email,property_title,start_time,end_time,calendar_event_id,status,created_at
0,1,Ali Raza,0300-9876543,cfiza8180@gmail.com,"10 Marla House, Bahria Town Lahore",2026-09-27T16:00:00,2026-09-27T17:00:00,1tc52mct3k69uru1vd5e7j1e84,cancelled,2026-09-25T02:18:26.003263
1,2,Bilal Khan,0300-5556667,cfiza8180@gmail.com,"1 Kanal House, DHA Phase 5 Lahore",2026-09-27T14:00:00,2026-09-27T15:00:00,56bjleeuq97hdc4pp78pfefuh0,booked,2026-09-25T02:21:02.151339


In [23]:
crm_log = pd.read_sql_query("SELECT * FROM crm_calls", conn)
crm_log.to_csv("crm_log.csv", index=False)
print(f"Saved {len(crm_log)} CRM entries to crm_log.csv")

Saved 2 CRM entries to crm_log.csv


In [24]:
summary = {
    "day": 4,
    "tasks_completed": [
        "Google Calendar integration (create + check availability)",
        "Gmail email automation",
        "Appointment management (book / reschedule / cancel)",
        "n8n webhook integration",
        "CRM logging (Google Sheets + SQLite)",
    ],
    "appointments_total": len(appts),
    "crm_entries_total": len(crm_log),
    "calendar_id": TEST_CALENDAR_ID,
    "n8n_webhook": N8N_WEBHOOK_URL,
}
with open("day4_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\n--- Day 4 COMPLETE ---")

{
  "day": 4,
  "tasks_completed": [
    "Google Calendar integration (create + check availability)",
    "Gmail email automation",
    "Appointment management (book / reschedule / cancel)",
    "n8n webhook integration",
    "CRM logging (Google Sheets + SQLite)"
  ],
  "appointments_total": 2,
  "crm_entries_total": 2,
  "calendar_id": "cfiza8180@gmail.com",
  "n8n_webhook": "http://localhost:5678/webhook/6901482b-d1ac-4d50-8a07-024ca415a718"
}

--- Day 4 COMPLETE ---


In [25]:
# conn.close()   # Comment out — aage aur cells ke liye open rakho
print("Notebook done. Connection open.")

Notebook done. Connection open.
